In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vit_timesformer_probe import TimeSformer, TimeSformerLinearProbe
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="fullshot"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:0' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-25 01:16:50,243]::3184030073::DEBUG::=============fullshot+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:0


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
mtcnn= MTCNN(image_size=224, margin=0, min_face_size= 20, thresholds=[0.6, 0.7, 0.7], post_process=True)
print(1)

1


In [5]:
train_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/train_set.dat","train")
print(1)
validation_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/valid_set.dat","valid")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,fullshot,tagdata,transform=None):
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20
lr = 3e-05
epochs = 50

# Multi-seed setting
SEEDS = [42, 123, 777, 1004, 2024]
patience = 10

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print(1)

print("UDIVA metric: MSE only (lower is better; no 1-MSE)")


1
UDIVA metric: MSE only (lower is better; no 1-MSE)


In [9]:
train_set_data = ChalearnDataset(
    fullshot=train_set_data[1],
    tagdata=train_set_data[2],
    transform=transform
)

val_set_data = ChalearnDataset(
    fullshot=validation_set_data[1],
    tagdata=validation_set_data[2],
    transform=transform
)

print(1)


1


In [10]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [11]:
# ============================================================
# TimeSformer Video-only Block-wise Linear Probe -- UDIVA / MSE
# ============================================================
#
# Probe targets:
#   block1 ... block12 : CLS token after each complete TimeSformer block
#   final_norm         : CLS token after final LayerNorm
#
# Backbone is completely frozen.
# Only independent Linear(768, 5) probe heads are optimized.
#
# IMPORTANT:
#   Metric = MSE (lower is better).
#   DO NOT use 1-MSE.
#   O/C/E/A/N are also evaluated directly with MSE.
# ============================================================

PROBE_SEED = 42
PROBE_LR = 1e-3
WEIGHT_DECAY = 1e-4
PROBE_EPOCHS = 50
PATIENCE = 10

BACKBONE_CHECKPOINT = "./save_timesformer_folder/model_best_multiseed.pth"


def set_probe_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def probe_seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_probe_seed(PROBE_SEED)

generator = torch.Generator()
generator.manual_seed(PROBE_SEED)

train_dataloader = DataLoader(
    dataset=train_set_data,
    batch_size=batchsz,
    shuffle=True,
    num_workers=num_workerssz,
    worker_init_fn=probe_seed_worker,
    generator=generator
)

val_dataloader = DataLoader(
    dataset=val_set_data,
    batch_size=batchsz,
    shuffle=False,
    num_workers=num_workerssz,
    worker_init_fn=probe_seed_worker,
    generator=generator
)

# ------------------------------------------------------------
# 1. Re-create exactly the same TimeSformer backbone
# ------------------------------------------------------------
backbone = TimeSformer().to(device)

checkpoint = torch.load(
    BACKBONE_CHECKPOINT,
    map_location=device
)

state_dict = (
    checkpoint["model"]
    if isinstance(checkpoint, dict) and "model" in checkpoint
    else checkpoint
)

backbone.load_state_dict(
    state_dict,
    strict=True
)

# ------------------------------------------------------------
# 2. Frozen backbone + independent linear heads
# ------------------------------------------------------------
model = TimeSformerLinearProbe(
    backbone=backbone,
    dim=768,
    num_outputs=5
).to(device)

backbone_trainable = sum(
    p.numel()
    for p in model.backbone.parameters()
    if p.requires_grad
)

probe_trainable = sum(
    p.numel()
    for p in model.probe_heads.parameters()
    if p.requires_grad
)

print(f"Backbone trainable parameters: {backbone_trainable}")
print(f"Probe trainable parameters:    {probe_trainable}")
print(f"Metric: MSE | Patience: {PATIENCE}")

assert backbone_trainable == 0, "Backbone must remain frozen."

criterion = nn.MSELoss().to(device)

optimizer = torch.optim.AdamW(
    model.probe_heads.parameters(),
    lr=PROBE_LR,
    weight_decay=WEIGHT_DECAY
)

probe_names = list(model.probe_heads.keys())

best_mse = {
    name: float("inf")
    for name in probe_names
}

best_epoch = {
    name: -1
    for name in probe_names
}

best_mean_val_mse = float("inf")
no_improve_count = 0

for epoch in range(PROBE_EPOCHS):

    # ========================================================
    # Train probe heads only
    # ========================================================
    model.train()

    train_sq_sum = {
        name: 0.0
        for name in probe_names
    }
    train_count = 0

    for fullshot, big_five_data in tqdm(
        train_dataloader,
        desc=f"Probe Train {epoch + 1}/{PROBE_EPOCHS}"
    ):
        fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
        big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)

        optimizer.zero_grad()

        outputs = model(fullshot)

        losses = {
            name: criterion(outputs[name], big_five_data)
            for name in probe_names
        }

        # Keep the original joint-probe training logic:
        # all independent probe-head losses are summed.
        total_loss = sum(losses.values())
        total_loss.backward()
        optimizer.step()

        scalar_count = big_five_data.numel()
        train_count += scalar_count

        for name in probe_names:
            train_sq_sum[name] += torch.square(
                outputs[name].detach() - big_five_data
            ).sum().item()

    train_mse = {
        name: train_sq_sum[name] / train_count
        for name in probe_names
    }

    # ========================================================
    # Validation
    # ========================================================
    model.eval()

    val_sq_sum = {
        name: 0.0
        for name in probe_names
    }

    # Per-trait squared-error sums for O/C/E/A/N.
    val_trait_sq_sum = {
        name: np.zeros(5, dtype=np.float64)
        for name in probe_names
    }

    val_count = 0
    val_sample_count = 0

    with torch.no_grad():

        for fullshot, big_five_data in tqdm(
            val_dataloader,
            desc=f"Probe Valid {epoch + 1}/{PROBE_EPOCHS}"
        ):
            fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
            big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)

            # Protect the batch dimension when the last batch has size 1.
            if big_five_data.ndim == 1:
                big_five_data = big_five_data.unsqueeze(0)

            outputs = model(fullshot)

            batch_samples = big_five_data.shape[0]
            val_sample_count += batch_samples
            val_count += big_five_data.numel()

            for name in probe_names:
                pred = outputs[name]
                sq_err = torch.square(pred - big_five_data)

                # Overall MSE numerator across all samples and five traits.
                val_sq_sum[name] += sq_err.sum().item()

                # O/C/E/A/N MSE numerators.
                val_trait_sq_sum[name] += (
                    sq_err.sum(dim=0).detach().cpu().numpy()
                )

    val_mse = {
        name: val_sq_sum[name] / val_count
        for name in probe_names
    }

    val_trait_mse = {
        name: val_trait_sq_sum[name] / val_sample_count
        for name in probe_names
    }

    # Each probe independently tracks its own lowest validation MSE.
    for name in probe_names:
        if val_mse[name] < best_mse[name]:
            best_mse[name] = val_mse[name]
            best_epoch[name] = epoch + 1

    # Global early stopping criterion = mean validation MSE
    # over all independent probe heads.
    mean_val_mse = float(
        np.mean([val_mse[name] for name in probe_names])
    )

    if mean_val_mse < best_mean_val_mse:
        best_mean_val_mse = mean_val_mse
        no_improve_count = 0
    else:
        no_improve_count += 1

    print("\n" + "-" * 132)
    print(
        f"Epoch {epoch + 1:03d} | "
        f"Mean Val MSE={mean_val_mse:.6f} | "
        f"Best Mean Val MSE={best_mean_val_mse:.6f} | "
        f"Patience={no_improve_count}/{PATIENCE}"
    )

    for name in probe_names:
        o_mse, c_mse, e_mse, a_mse, n_mse = val_trait_mse[name]
        print(
            f"{name:<12} | "
            f"Train MSE {train_mse[name]:.6f} | "
            f"Val MSE {val_mse[name]:.6f} | "
            f"Best MSE {best_mse[name]:.6f} | "
            f"O {o_mse:.6f} | C {c_mse:.6f} | E {e_mse:.6f} | "
            f"A {a_mse:.6f} | N {n_mse:.6f}"
        )

    if no_improve_count >= PATIENCE:
        print(
            f"\nEarly stopping at epoch {epoch + 1}. "
            f"Best mean validation MSE={best_mean_val_mse:.6f} | "
            f"Patience={no_improve_count}/{PATIENCE}"
        )
        break

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# Final Probe Summary -- MSE only
# ============================================================

print("\n" + "=" * 72)
print("FINAL TIMESFORMER VIDEO-ONLY LINEAR PROBE RESULTS -- UDIVA / MSE")
print("=" * 72)
print(
    f"{'Representation':<18}"
    f"{'Best Epoch':>12}"
    f"{'Best MSE':>16}"
)
print("-" * 72)

for name in probe_names:
    print(
        f"{name:<18}"
        f"{best_epoch[name]:>12d}"
        f"{best_mse[name]:>16.6f}"
    )

print("=" * 72)

del model
del backbone
del optimizer
del train_dataloader
del val_dataloader

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131
Backbone trainable parameters: 0
Probe trainable parameters:    49985
Metric: MSE | Patience: 10


Probe Valid 1/50: 100%|██████████| 191/191 [00:18<00:00, 10.60it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 001 | Mean Val MSE=1.408031 | Best Mean Val MSE=1.408031 | Patience=0/10
block1       | Train MSE 0.168906 | Val MSE 1.569642 | Best MSE 1.569642 | O 1.627146 | C 1.656943 | E 1.029230 | A 2.064450 | N 1.470439
block2       | Train MSE 0.154604 | Val MSE 1.461273 | Best MSE 1.461273 | O 1.532540 | C 1.535737 | E 0.976217 | A 1.999949 | N 1.261920
block3       | Train MSE 0.143442 | Val MSE 1.385373 | Best MSE 1.385373 | O 1.471886 | C 1.423689 | E 0.963046 | A 1.841014 | N 1.227230
block4       | Train MSE 0.135236 | Val MSE 1.383667 | Best MSE 1.383667 | O 1.445221 | C 1.443399 | E 0.957742 | A 1.758315 | N 1.313656
block5       | Train MSE 0.128277 | Val MSE 1.377603 | Best MSE 1.377603 | O 1.419984 | C 1.438988 | E 0.953346 | A 1.772480 | N 1.303220
block6       | Train MSE 0.130563 | Val MSE 1.380548 | Best MSE 1.380548 | O 1.457673 | C 1.43969

Probe Valid 2/50: 100%|██████████| 191/191 [00:17<00:00, 10.62it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 002 | Mean Val MSE=1.429881 | Best Mean Val MSE=1.408031 | Patience=1/10
block1       | Train MSE 0.117176 | Val MSE 1.670936 | Best MSE 1.569642 | O 2.012811 | C 1.504304 | E 1.044326 | A 2.089043 | N 1.704193
block2       | Train MSE 0.116309 | Val MSE 1.596364 | Best MSE 1.461273 | O 1.761214 | C 1.794777 | E 0.945682 | A 2.143977 | N 1.336171
block3       | Train MSE 0.112668 | Val MSE 1.480720 | Best MSE 1.385373 | O 1.600798 | C 1.685869 | E 0.944344 | A 1.958491 | N 1.214099
block4       | Train MSE 0.107754 | Val MSE 1.422606 | Best MSE 1.383667 | O 1.445269 | C 1.627186 | E 0.995377 | A 1.827839 | N 1.217357
block5       | Train MSE 0.104542 | Val MSE 1.399531 | Best MSE 1.377603 | O 1.386396 | C 1.571303 | E 1.010679 | A 1.808815 | N 1.220460
block6       | Train MSE 0.101884 | Val MSE 1.394639 | Best MSE 1.380548 | O 1.402366 | C 1.53153

...


------------------------------------------------------------------------------------------------------------------------------------
Epoch 009 | Mean Val MSE=1.423929 | Best Mean Val MSE=1.408031 | Patience=8/10
block1       | Train MSE 0.076435 | Val MSE 1.590696 | Best MSE 1.569642 | O 1.560670 | C 1.600747 | E 1.053520 | A 2.094383 | N 1.644159
block2       | Train MSE 0.072984 | Val MSE 1.550993 | Best MSE 1.461273 | O 1.484522 | C 1.609080 | E 0.889292 | A 2.151360 | N 1.620711
block3       | Train MSE 0.072935 | Val MSE 1.473850 | Best MSE 1.385373 | O 1.528927 | C 1.545055 | E 0.893038 | A 1.944770 | N 1.457459
block4       | Train MSE 0.071706 | Val MSE 1.412641 | Best MSE 1.383667 | O 1.493423 | C 1.455295 | E 0.907127 | A 1.819308 | N 1.388053
block5       | Train MSE 0.071677 | Val MSE 1.392009 | Best MSE 1.377603 | O 1.470010 | C 1.417576 | E 0.892481 | A 1.815115 | N 1.364862
block6       | Train MSE 0.071429 | Val MSE 1.403265 | Best MSE 1.380548 | O 1.510467 | C 1.41756

Probe Valid 10/50: 100%|██████████| 191/191 [00:19<00:00,  9.86it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 010 | Mean Val MSE=1.487564 | Best Mean Val MSE=1.408031 | Patience=9/10
block1       | Train MSE 0.076306 | Val MSE 1.475266 | Best MSE 1.475266 | O 1.515849 | C 1.458934 | E 0.977659 | A 1.932468 | N 1.491421
block2       | Train MSE 0.075443 | Val MSE 1.556910 | Best MSE 1.461273 | O 1.514124 | C 1.719744 | E 1.079561 | A 2.030524 | N 1.440600
block3       | Train MSE 0.074134 | Val MSE 1.549987 | Best MSE 1.385373 | O 1.490181 | C 1.751272 | E 1.190787 | A 1.904607 | N 1.413090
block4       | Train MSE 0.071705 | Val MSE 1.504928 | Best MSE 1.383667 | O 1.451557 | C 1.658699 | E 1.170793 | A 1.821572 | N 1.422018
block5       | Train MSE 0.071536 | Val MSE 1.496831 | Best MSE 1.377603 | O 1.430821 | C 1.628091 | E 1.152983 | A 1.845257 | N 1.427003
block6       | Train MSE 0.071169 | Val MSE 1.490922 | Best MSE 1.380548 | O 1.445316 | C 1.60226

Probe Valid 11/50: 100%|██████████| 191/191 [00:19<00:00,  9.83it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 011 | Mean Val MSE=1.484350 | Best Mean Val MSE=1.408031 | Patience=10/10
block1       | Train MSE 0.074757 | Val MSE 1.569425 | Best MSE 1.475266 | O 1.497235 | C 1.740861 | E 0.987315 | A 1.942862 | N 1.678849
block2       | Train MSE 0.073555 | Val MSE 1.529998 | Best MSE 1.461273 | O 1.458511 | C 1.630412 | E 1.039512 | A 1.890859 | N 1.630698
block3       | Train MSE 0.074327 | Val MSE 1.501213 | Best MSE 1.385373 | O 1.465120 | C 1.676991 | E 1.084395 | A 1.830481 | N 1.449079
block4       | Train MSE 0.073255 | Val MSE 1.482797 | Best MSE 1.383667 | O 1.409968 | C 1.726348 | E 1.133698 | A 1.768588 | N 1.375382
block5       | Train MSE 0.072001 | Val MSE 1.466674 | Best MSE 1.377603 | O 1.388658 | C 1.705884 | E 1.110507 | A 1.780202 | N 1.348122
block6       | Train MSE 0.071014 | Val MSE 1.472619 | Best MSE 1.380548 | O 1.401211 | C 1.7203